In [50]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
import warnings
warnings.filterwarnings("ignore")

In [4]:
df=pd.read_csv("train.csv")

In [9]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [38]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## let's drop

In [39]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [40]:
# Step 1 -> train/test/split
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [41]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [42]:
x_train.sample(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
779,1,female,43.0,0,1,211.3375,S
741,1,male,36.0,1,0,78.8500,S
856,1,female,45.0,1,1,164.8667,S
187,1,male,45.0,0,0,26.5500,S
93,3,male,26.0,1,2,20.5750,S


## Agar aap transformers= word nahi likhna chahte, to positional argument ke through likh sakte ho:

In [43]:
# imputation transfromer
trf1=ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),# stragtegy is equal to mean
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6]),# [2] column index and position
],remainder='passthrough')

In [44]:
# one hot encoding
trf2=ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6]) # drop:first don't do because decision tree classifier
],remainder='passthrough')

In [45]:
# Scaling
trf3=ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10)) # all column apply transformation
])

In [52]:
# feature selection
trf4=SelectKBest(score_func=chi2,k=8)

In [53]:
# train the model
trf5= DecisionTreeClassifier()

## Create Pipeline

In [57]:
# pipeline 
pipe=Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5),
])

or 
### Alternate Syntax
pipe=make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [58]:
# train
pipe.fit(x_train,y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [59]:
# Display Pipeline
from sklearn import set_config
set_config(display='diagram')

## Explore the pipeline

In [60]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x000001B5328C7880>),
 'trf5': DecisionTreeClassifier()}

In [65]:
pipe.named_steps['trf1']

,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [63]:
pipe.named_steps['trf1'].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 5])]

In [70]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([29.49884615])

In [71]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [77]:
# predict
y_pred=pipe.predict(x_test)

In [78]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6256983240223464

In [79]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

## Cross Validation using Pipeline

In [80]:
 # cross validation using_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,x_train,y_train,cv=5,scoring='accuracy').mean()

np.float64(0.6391214419383433)

## GridSearch using Pipeline

In [84]:
# gridsearchcv
params={
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [86]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(pipe,params,cv=5,scoring='accuracy')
grid.fit(x_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf5__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"


In [94]:
grid.best_score_

np.float64(0.6391214419383433)

In [89]:
grid.best_params_

{'trf5__max_depth': 2}

## Exporting the Pipeline
production ready code

In [90]:
# Export
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))

In [91]:
pipe=pickle.load(open('pipe.pkl','rb'))

input

 Pclass     Sex      Age     SibSp     Parch     Fare     Embarked
  2        male     31       0          0        10.5        s

In [92]:
# assume user input 
test_input2=np.array([2,'male',31.0,0,0,10.5,'s'],dtype=object).reshape(1,7)

In [93]:
pipe.predict(test_input2)

array([0])